# 手撕 Repetition Penalty + No-Repeat N-Gram

## 背景
Repetition Penalty：已出现 token 的 logits 除以 penalty（降权）。
No-Repeat N-Gram：禁止已出现的 n-gram 继续生成。

## 考察点
- repetition penalty 的方向（除法而非减法）
- no-repeat-ngram 的实现（查表）
- HuggingFace 中的实现方式

In [ ]:
import torch

def apply_repetition_penalty(logits, generated_ids, penalty=1.2):
    # logits: (vocab_size,), generated_ids: list[int]
    for token_id in set(generated_ids):
        if logits[token_id] > 0:
            logits[token_id] /= penalty
        else:
            logits[token_id] *= penalty
    return logits

def ban_repeated_ngrams(logits, generated_ids, n=3):
    # 禁止已出现 n-gram 的下一个 token
    if len(generated_ids) < n - 1:
        return logits
    # 当前 n-1 前缀
    prefix = tuple(generated_ids[-(n-1):])
    # 查找所有已出现的 n-gram
    banned = set()
    for i in range(len(generated_ids) - n + 1):
        if tuple(generated_ids[i:i+n-1]) == prefix:
            banned.add(generated_ids[i + n - 1])
    logits[list(banned)] = float('-inf')
    return logits

In [ ]:
# 验证 repetition penalty
torch.manual_seed(42)
logits = torch.tensor([2.0, 1.5, 3.0, 0.5, 1.0])
generated = [0, 2, 2, 1]  # token 2 已出现两次
penalized = apply_repetition_penalty(logits.clone(), generated, penalty=1.5)
# token 2 的 logit 应被降低
assert penalized[2] < logits[2], "已出现 token 应被降权"
print(f"原始 logits: {logits.tolist()}")
print(f"惩罚后 logits: {penalized.tolist()}")
# 验证 no-repeat-3gram
logits2 = torch.tensor([1.0, 1.0, 1.0, 1.0, 1.0])
generated2 = [0, 1, 2, 0, 1]  # 3-gram (0,1,2) 和 (0,1) 已出现
banned = ban_repeated_ngrams(logits2.clone(), generated2, n=3)
assert banned[2] == float('-inf'), "token 2 应被禁止（避免 0,1,2 重复）"
print(f"no-repeat-3gram 后: {banned.tolist()}")
print("✅ Repetition Penalty + No-Repeat N-Gram 验证通过")